# Hands-on Session 2: Running DGAT Protein Inference Workflows

**Time:** 11:05-11:35  
**Lead:** Zeyuan

Goals:
- Prepare transcript and spatial inputs for DGAT-style inference.
- Run the official DGAT workflow or load pretrained DGAT predictions.
- Generate inferred spatial protein landscapes.
- Visualize predicted protein expression patterns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad, find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table, run_demo_dgat_inference, write_prediction_artifact
from dgat_tutorial.plotting import plot_spatial_feature

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "data" / "raw"
processed_dir = repo_root / "data" / "processed"
fig_dir = repo_root / "results" / "figures"
processed_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# Default: load predictions generated by official DGAT and distributed by the organizers.
# The only alternative here is an explicitly labeled out-of-fold ridge baseline (not DGAT).
prediction_mode = "precomputed"  # choose: "precomputed" or "baseline"
allow_synthetic_data = False

## Load Tutorial Data

In [ ]:
dgat_h5ad_pair = find_dgat_h5ad_pair(data_dir)
dgat_h5ad = find_dgat_h5ad(data_dir)
dataset = load_tutorial_data(data_dir, allow_demo=allow_synthetic_data)
if dgat_h5ad_pair:
    print(f"Loaded DGAT AnnData files: RNA={dgat_h5ad_pair[0]}, ADT={dgat_h5ad_pair[1]}")
else:
    print(f"Loaded DGAT AnnData file: {dgat_h5ad}" if dgat_h5ad else "Loaded synthetic fallback data")

spots = dataset.spots
transcripts = dataset.transcripts
proteins = dataset.proteins

print(transcripts.shape, proteins.shape)

## Prepare DGAT Inputs

For the live tutorial, use the official DGAT repository as the source of model code: https://github.com/osmanbeyoglulab/DGAT

The original DGAT workflow reads `.h5ad` data directly. This notebook aligns the observations in memory and does not export the full transcript/protein matrices to CSV; those redundant writes can be slow and memory-intensive.

Official DGAT inference requires the separate Python 3.11 environment in `environment-dgat-cpu.yml`. Run it before the session with `scripts/run_official_dgat_prediction.py`, then distribute its output as `data/raw/dgat_predictions.csv`.

In [ ]:
common_spots = spots.index.intersection(transcripts.index).intersection(proteins.index)
spots = spots.loc[common_spots]
transcripts = transcripts.loc[common_spots]
proteins = proteins.loc[common_spots]
print(f"Prepared {len(common_spots)} aligned spots")

## Run or Load DGAT Predictions

Default participant path: load the organizer-provided official DGAT predictions from `data/raw/dgat_predictions.csv`.

The notebook does **not** silently fit a model to the observed proteins. For an environment check only, set `prediction_mode = "baseline"`; this runs an explicitly labeled out-of-fold ridge baseline, so each row is predicted by a model that did not train on that row. The baseline is not DGAT and must not be presented as a DGAT result.

In [ ]:
prediction_path = data_dir / "dgat_predictions.csv"

if prediction_mode == "precomputed":
    if not prediction_path.exists():
        raise FileNotFoundError(
            f"Missing {prediction_path}. Download the organizer-provided official DGAT predictions before the tutorial. "
            "Organizers: generate them in the separate Python 3.11 DGAT environment."
        )
    predicted_proteins = load_prediction_table(str(prediction_path))
    input_metadata = load_prediction_metadata(prediction_path)
    method = str(input_metadata["method"]) if input_metadata else "precomputed_dgat"
    source = str(input_metadata["source"]) if input_metadata else str(prediction_path)
    evaluation_note = (
        str(input_metadata["evaluation_note"])
        if input_metadata
        else "Confirm the organizer generated this table without fitting on the evaluation proteins."
    )
    print("Loaded precomputed official DGAT predictions")
elif prediction_mode == "baseline":
    predicted_proteins = run_demo_dgat_inference(transcripts, proteins)
    method = "out_of_fold_ridge_baseline"
    source = "observed tutorial RNA/protein matrices"
    evaluation_note = "Not DGAT; every row is out-of-fold, avoiding in-sample evaluation."
    print("Generated out-of-fold ridge baseline (not DGAT)")
else:
    raise ValueError("prediction_mode must be 'precomputed' or 'baseline'")

output_path = processed_dir / "predicted_proteins.csv"
write_prediction_artifact(
    predicted_proteins, output_path, method=method, source=source, evaluation_note=evaluation_note
)
predicted_proteins.head()

## Visualize Inferred Protein Landscapes

In [ ]:
proteins_to_plot = list(predicted_proteins.columns[:4])
fig, axes = plt.subplots(2, 2, figsize=(9, 8))

for ax, protein in zip(axes.ravel(), proteins_to_plot):
    plot_spatial_feature(spots, predicted_proteins[protein], f"Predicted {protein}", ax=ax)

plt.tight_layout()
plt.savefig(fig_dir / "session02_predicted_proteins.png", dpi=160)
plt.show()

## Checkpoint

Participants should now have a predicted protein matrix, a provenance sidecar, and spatial plots for Session 3. State the printed method whenever presenting the results.